In [1]:
# -----######-----###### MAIN IMPORTS -----######-----######
import os, subprocess, shlex, re, pandas as pd

def _tqm_bar(step, total, label):
    width = 28
    frac = step / float(total)
    filled = int(width * frac)
    bar = "█" * filled + " " * (width - filled)
    print(f"TQM | {label}: {int(frac*100):3d}%|{bar}| {step}/{total}")

# ----- helpers -----
def _safe_filename(x):
    return re.sub(r'[\\/:*?"<>|]+', "_", str(x))

def _parse_genre_from_name(name):
    """
    Extracts the 'genre string' as everything BEFORE the BPM token.
    Examples:
      'TECH_HOUSE_126'                -> 'TECH HOUSE'
      'MINIMAL_DEEP_HOUSE_124_2'      -> 'MINIMAL DEEP HOUSE'
      'BASS-CLUB_150'                 -> 'BASS CLUB'
      'OPEN_FORMAT_NIGGD'             -> 'OPEN FORMAT NIGGD'  (no bpm -> whole name)
    """
    base = str(name).strip().replace("-", "_")
    parts = [p for p in re.split(r"_+", base) if p != ""]
    bpm_idx = None
    for i, p in enumerate(parts):
        if re.fullmatch(r"\d{2,3}", p):  # treat a pure 2–3 digit token as BPM
            bpm_idx = i
            break
    genre_parts = parts[:bpm_idx] if bpm_idx is not None else parts
    # collapse to spaces, keep uppercase (as your style)
    genre_str = " ".join(genre_parts).strip()
    return genre_str if genre_str else "UNKNOWN"

# ==============================
#  -----######-----###### 
#  _silence_1710_numberedgenre_GET_df_mp3
#  -----######-----######
# ==============================
def _silence_1710_numberedgenre_GET_df_mp3(
    names,
    out_dir,
    duration_sec=30,
    sr=44100,
    channels=2,
    bitrate_kbps=192
):
    """
    Generate numbered silent MP3s (default 30 sec) from a list of names.
    - File names prefixed with 001_, 002_, ...
    - ID3 title  = '--++__{name}_____++++++-------'
    - ID3 album  = basename(out_dir)
    - ID3 genre  = part of name BEFORE the BPM token (underscores/hyphens -> spaces)
    """
    os.makedirs(out_dir, exist_ok=True)
    album_name = os.path.basename(os.path.normpath(out_dir))

    rows = []
    total = len(names)
    print("────────────────────────────────────────────────────────────")
    print(f"🎧 Creating {total} silent MP3s → {out_dir}")
    print(f"💿 Album tag: '{album_name}'")
    print("────────────────────────────────────────────────────────────")

    for i, raw in enumerate(names, start=1):
        _tqm_bar(i, total, "Batch Progress")

        title_src = str(raw).strip() or f"silent_{i:03d}"
        genre_tag = _parse_genre_from_name(title_src)

        safe = _safe_filename(title_src)
        num_prefix = f"{i:03d}_"
        file_name = f"{num_prefix}{safe}.mp3"
        out_path = os.path.join(out_dir, file_name)

        id3_title = f"--++__{title_src}_____++++++-------"

        cmd = (
            f'ffmpeg -y -f lavfi -t {int(duration_sec)} '
            f'-i anullsrc=channel_layout={"stereo" if channels==2 else "mono"}:sample_rate={int(sr)} '
            f'-b:a {int(bitrate_kbps)}k '
            f'-metadata title="{id3_title}" '
            f'-metadata album="{album_name}" '
            f'-metadata genre="{genre_tag}" '
            f'{shlex.quote(out_path)}'
        )

        status, err = "ok", ""
        try:
            subprocess.run(shlex.split(cmd), stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, check=True)
        except Exception as e:
            status, err = "fail", str(e)

        rows.append({
            "num": i,
            "title_in": title_src,
            "genre_tag": genre_tag,
            "file_name": file_name,
            "Path": out_path,
            "status": status,
            "err": err
        })

    return pd.DataFrame(rows)


In [3]:
names = [
    "MINIMAL_DEEP_TECH_124",
    "MINIMAL_DEEP_TECH_127",
    "MINIMAL_DEEP_HOUSE_122",
    "MINIMAL_DEEP_HOUSE_124",
    "MINIMAL_DEEPHOUSE_126",
    "HOUSE_122_",
    "HOUSE_124_",
    "HOUSE_125_",
    "HOUSE_125_2",
    "HOUSE_126_",
    "HOUSE_127_",
    "HOUSE_127_2",
    "HOUSE_128_",
    "HOUSE_129_",
    "HOUSE_130_",
    "HOUSE_131_",
    "DEEP_HOUSE_122",
    "DEEP_HOUSE_123",
    "DEEP_HOUSE_124",
    "DEEP_HOUSE_125",
    "DEEP_HOUSE_126",
    "DEEP_HOUSE_128",
    "NU_DISCO_122",
    "NU_DISCO_125",
    "90S_HOUSE_126",
    "FUNKY_JACKIN_HOUSE_126",
    "TECH_HOUSE_124",
    "TECH_HOUSE_124_2",
    "TECH_HOUSE_125",
    "TECH_HOUSE_126",
    "TECH_HOUSE_127",
    "TECH_HOUSE_128",
    "TECH_HOUSE_128_2",
    "ELECTRO_132",
    "LATIN_123",
    "LATIN_125",
    "LATIN_135",
    "BREAKS_UK_GARAGE_133",
    "BASS_CLUB_132",
    "BASS_CLUB_135",
    "BASS_CLUB_150",
    "OPEN_FORMAT_NIGGD"
]


# 👇 Output folder
out_dir = "/Users/yerik/Music/_1_NEW_SOURCE/__silent__"
df = _silence_1710_numberedgenre_GET_df_mp3(names, out_dir)


────────────────────────────────────────────────────────────
🎧 Creating 42 silent MP3s → /Users/yerik/Music/_1_NEW_SOURCE/__silent__
💿 Album tag: '__silent__'
────────────────────────────────────────────────────────────
TQM | Batch Progress:   2%|                            | 1/42
TQM | Batch Progress:   4%|█                           | 2/42
TQM | Batch Progress:   7%|██                          | 3/42
TQM | Batch Progress:   9%|██                          | 4/42
TQM | Batch Progress:  11%|███                         | 5/42
TQM | Batch Progress:  14%|████                        | 6/42
TQM | Batch Progress:  16%|████                        | 7/42
TQM | Batch Progress:  19%|█████                       | 8/42
TQM | Batch Progress:  21%|██████                      | 9/42
TQM | Batch Progress:  23%|██████                      | 10/42
TQM | Batch Progress:  26%|███████                     | 11/42
TQM | Batch Progress:  28%|████████                    | 12/42
TQM | Batch Progress:  30%|██████

In [6]:
22*10 +22

242